In [44]:
from typing import List
import stim
from stimcirq import stim_circuit_to_cirq_circuit
import cirq
import openfermion as of
from encoded.code_extension import encoding_unitary_for_new_stabilizer
from encoded.utils import cirq_pauli_string_to_stim

In [45]:
n = 7
def extended_repetition_generators(n: int) -> List[stim.PauliString]:
    generators = []
    for i in range(n-1):
        pauli_str = 'I' * i + 'Z' * 2 + 'I' * (n - i - 2)
        assert len(pauli_str) == n
        generators.append(stim.PauliString(pauli_str))
    generators.append(stim.PauliString('X' * n))
    return generators

In [46]:
generators = extended_repetition_generators(n)
for generator in generators:
    print(generator)

+ZZ_____
+_ZZ____
+__ZZ___
+___ZZ__
+____ZZ_
+_____ZZ
+XXXXXXX


In [47]:
for gi in generators:
    for gj in generators:
        assert gi.commutes(gj)

In [48]:
def all_single_qubit_errs(n: int) -> List[stim.PauliString]:
    """Returns the set of all single-qubit Pauli errors."""

    errs = []
    for i in range(n):
        for p in ['X', 'Y', 'Z']:
            pauli_str = "I" * i + p + 'I' * (n - i - 1)
            assert len(pauli_str) == n
            errs.append(stim.PauliString(pauli_str))
    return errs

In [49]:
errors = all_single_qubit_errs(n)

In [50]:
number_false = 0
number_checked = 0
for i, ei in enumerate(errors):
    for j in range(i):
        number_checked += 1
        ej = errors[j]
        e = ei * ej
        commutators = []
        for generator in generators:
            comm = e.commutes(generator)
            commutators.append(comm)
        has_anticommuting_operator = any([not b for b in commutators])
        if has_anticommuting_operator:
            number_false += 1
        if not has_anticommuting_operator:
            print(f"{ei} * {ej} = {e}, {commutators} {has_anticommuting_operator} ")
print(f"{number_false}/{number_checked} operators anticommute.")

+_Z_____ * +Z______ = +ZZ_____, [True, True, True, True, True, True, True] False 
+__Z____ * +Z______ = +Z_Z____, [True, True, True, True, True, True, True] False 
+__Z____ * +_Z_____ = +_ZZ____, [True, True, True, True, True, True, True] False 
+___Z___ * +Z______ = +Z__Z___, [True, True, True, True, True, True, True] False 
+___Z___ * +_Z_____ = +_Z_Z___, [True, True, True, True, True, True, True] False 
+___Z___ * +__Z____ = +__ZZ___, [True, True, True, True, True, True, True] False 
+____Z__ * +Z______ = +Z___Z__, [True, True, True, True, True, True, True] False 
+____Z__ * +_Z_____ = +_Z__Z__, [True, True, True, True, True, True, True] False 
+____Z__ * +__Z____ = +__Z_Z__, [True, True, True, True, True, True, True] False 
+____Z__ * +___Z___ = +___ZZ__, [True, True, True, True, True, True, True] False 
+_____Z_ * +Z______ = +Z____Z_, [True, True, True, True, True, True, True] False 
+_____Z_ * +_Z_____ = +_Z___Z_, [True, True, True, True, True, True, True] False 
+_____Z_ * +__Z_